# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated: commit, val_roi, val_brier, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_roi"]   = pd.to_numeric(df["val_roi"],   errors="coerce")
df["val_brier"] = pd.to_numeric(df["val_brier"], errors="coerce")
df["status"]    = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH",   0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    roi   = row["val_roi"]
    brier = row["val_brier"]
    desc  = row["description"]
    print(f"  #{i:3d}  roi={roi:.6f}  brier={brier:.4f}  {desc}")

## Val ROI Over Time

Track how the best (kept) val_roi evolves as experiments progress.
The running maximum shows the frontier — the best ROI achieved so far.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Left: ROI over time ---
ax = axes[0]
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

baseline_roi = valid.loc[0, "val_roi"] if len(valid) > 0 else 0.0

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_roi"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_roi"],
           c="#2ecc71", s=50, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
kept_roi  = valid.loc[kept_mask, "val_roi"]
running_max = kept_roi.cummax()
ax.step(kept_idx, running_max, where="post",
        color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, roi in zip(kept_idx, kept_roi):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, roi),
                textcoords="offset points", xytext=(6, 6),
                fontsize=8, color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept  = len(df[df["status"] == "KEEP"])
ax.axhline(0, color="red", linewidth=1, linestyle="--", alpha=0.5, label="Break-even (ROI=0)")
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation ROI (higher is better)", fontsize=12)
ax.set_title(f"ROI Progress: {n_total} Experiments, {n_kept} Kept", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# --- Right: Brier score over time ---
ax2 = axes[1]
disc_b = valid[valid["status"] == "DISCARD"]
ax2.scatter(disc_b.index, disc_b["val_brier"],
            c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")
ax2.scatter(kept_v.index, kept_v["val_brier"],
            c="#3498db", s=50, zorder=4, label="Kept",
            edgecolors="black", linewidths=0.5)

kept_brier   = valid.loc[kept_mask, "val_brier"]
running_min_b = kept_brier.cummin()
ax2.step(kept_idx, running_min_b, where="post",
         color="#2980b9", linewidth=2, alpha=0.7, zorder=3, label="Running best")

ax2.set_xlabel("Experiment #", fontsize=12)
ax2.set_ylabel("Validation Brier Score (lower is better)", fontsize=12)
ax2.set_title("Brier Score Progress", fontsize=13)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_roi = df.iloc[0]["val_roi"] if len(df) > 0 else float('nan')
best_roi     = kept["val_roi"].max() if len(kept) > 0 else float('nan')
best_row     = kept.loc[kept["val_roi"].idxmax()] if len(kept) > 0 else None

print(f"Baseline val_roi:  {baseline_roi:.6f}")
print(f"Best val_roi:      {best_roi:.6f}")
if baseline_roi != 0:
    print(f"Total improvement: {best_roi - baseline_roi:+.6f} ({(best_roi - baseline_roi) / abs(baseline_roi) * 100:.2f}%)")
if best_row is not None:
    print(f"Best experiment:   {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: roi={row['val_roi']:.6f}  brier={row['val_brier']:.4f}  {desc}")

## Top Hits (Kept Experiments by ROI Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_roi"] = kept["val_roi"].shift(1)
kept["delta"]    = kept["val_roi"] - kept["prev_roi"]  # positive = improvement

hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'ROI':>10}  {'Brier':>8}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_roi']:.6f}  {row['val_brier']:.4f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")